# Feature Engineering

## Team Selection

Choose the team, Grand Prix, and season that have already been processed. These parameters define the exact comparison window for the analysis.

In [1]:
team = "Aston Martin"
grand_prix = "Japanese Grand Prix"
year = 2025

## Imports and Configuration

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import numpy as np
from pathlib import Path
import fastf1

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src import load_data, preprocess_data, explore_data, feature_engineering

## Load Processed Data

Read the cached processed laps for both teammates so the notebook can work from already cleaned telemetry.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = load_data.get_drivers(session, team)
segment = load_data.get_segment(session, driver_1, driver_2)
corners = load_data.get_corners(session)

print(corners)

try:
    lap_1, lap_2 = preprocess_data.load_processed_data(driver_1, driver_2, year, grand_prix, segment)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (598.78 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']


    Number     Distance
0        1   718.865490
1        2   830.361033
2        3  1063.112431
3        4  1197.159026
4        5  1364.076590
5        6  1521.509446
6        7  1841.732500
7        8  2265.842642
8        9  2412.668486
9       10  2743.644082
10      11  2889.668982
11      12  3146.600553
12      13  3784.834879
13      14  3947.333109
14      15  5000.455312
15      16  5379.614914
16      17  5461.515861
17      18  5628.377564

Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Braking features

In [4]:
try:
    braking_zones_1 = feature_engineering.analyze_braking_zones(lap_1["Interpolated"])
    braking_zones_2 = feature_engineering.analyze_braking_zones(lap_2["Interpolated"])
except Exception as e:
    print(f"Braking zones analysis failed: {e}")

try:
    braking_zones_1 = feature_engineering.assign_braking_zones_to_corners(braking_zones_1, corners)
    braking_zones_2 = feature_engineering.assign_braking_zones_to_corners(braking_zones_2, corners)
except Exception as e:
    print(f"Assignment of braking zones to corners failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

Braking zones for ALO:

:    StartDistance  EndDistance  BrakingDuration  BrakingDistance  EntrySpeed  \
0     679.970542   782.977661         1.512405       103.007119  312.018048   
1    1469.691790  1512.611423         0.713592        42.919633  239.000000   
2    2233.661258  2285.164818         0.719429        51.503560  282.821207   
3    2345.252304  2413.923717         1.426009        68.671413  249.141365   
4    2791.616487  2877.455754         2.424912        85.839266  260.800903   
5    3718.680561  3787.351974         0.976355        68.671413  282.831508   
6    3856.023387  3924.694800         1.241569        68.671413  225.757158   
7    5255.203424  5341.042690         2.124844        85.839266  267.099759   

   MinimumSpeed  Corner  CornerDistance  
0    204.135603     1.0      718.865490  
1    208.819108     6.0     1521.509446  
2    262.830452     8.0     2265.842642  
3    142.311138     9.0     2412.668486  
4     79.115621    11.0     2889.668982  
5    238.5